# 05 - Interpretability

**Report stage:** Interpretability (Grad-CAM++ and slice attention)

Grad-CAM++ for the CNN models and slice-attention visualisation for the
attention-pooled models, to check whether predictions focus on clinically
relevant anatomy. The reusable functions live in `src/interpretability.py`;
this notebook is a worked example that loads a trained checkpoint and renders a
few exams.

> Renumbered from `06_interpretability.ipynb` after the obsolete cross-validation
> notebook was archived (see `archive/README.md`).

**Figures are written to** `for-gpu/results/figures/06_interpretability/`
(`config.FIGURE_DIRS["interpretability"]`).

## How to use
1. Run the **bootstrap** cell.
2. Load a trained checkpoint from `config.CHECKPOINTS_DIR`.
3. Render Grad-CAM++ / slice-attention overlays on a few exams.

In [ ]:
# ============================================================
# Colab bootstrap - run this cell first.
# ============================================================
import sys
from pathlib import Path

# Mount Google Drive when running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Locate the "codes" folder (the one containing src/ and notebooks/).
if IN_COLAB:
    # TODO: set this to where you uploaded the "codes" folder on Drive.
    CODES_DIR = Path('/content/drive/MyDrive/AI-for-MIA/codes')
else:
    # Local run: this notebook lives in codes/notebooks/, so go up one level.
    CODES_DIR = Path.cwd().parent

assert (CODES_DIR / 'src').exists(), (
    f"Could not find src/ under {CODES_DIR}. "
    "Edit CODES_DIR above to point to your 'codes' folder."
)

# Make `import src...` work.
if str(CODES_DIR) not in sys.path:
    sys.path.insert(0, str(CODES_DIR))

# Auto-reload edited src/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

# Shared config resolves the data folder (sibling of "codes").
from src import config
print('CODES_DIR:', config.CODES_DIR)
print('DATA_DIR :', config.DATA_DIR)
assert config.DATA_DIR.exists(), (
    f"Data folder not found at {config.DATA_DIR}. Put the MRNet 'data' folder "
    "next to 'codes', or set os.environ['MRNET_DATA_DIR'] before importing config."
)
print('Setup OK - data folder found.')


In [ ]:
import torch
import matplotlib.pyplot as plt

from src.data_pipeline import build_dataloaders
from src.model_factory import build_model
from src import interpretability as interp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Load a trained checkpoint (inference-ready: build_config embedded) -----
# Any variant trained by the GPU runners works. We use the pre-tuned
# DenseNet121+CBAM; each checkpoint records how to rebuild its own model.
ckpt_path = (config.CHECKPOINTS_DIR / "densenet121_strong_cbam_gpu"
             / "best_densenet121_strong_cbam.pth")
assert ckpt_path.exists(), (
    f"No checkpoint at {ckpt_path}. Train one first (notebook 02 or "
    "`for-gpu/run_cnn.py --backbone densenet121 --use-cbam`).")

ck = torch.load(ckpt_path, map_location="cpu")
bc = ck["build_config"]
model = build_model(backbone=bc["backbone"], use_cbam=bc.get("use_cbam", False),
                    dropout=bc.get("dropout", 0.0), use_checkpoint=False)
# Training validated BatchNorm in batch-stat mode; replicate it so the saved
# weights (no BN running buffers) load strictly. See run_cnn.use_batch_stat_bn.
if not bc.get("bn_running_stats", False):
    for m in model.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
            m.track_running_stats = False
            m.running_mean = None; m.running_var = None; m.num_batches_tracked = None
model.load_state_dict(ck["model_state_dict"])
model = model.to(device).eval()
print(f"loaded {bc['backbone']} (cbam={bc.get('use_cbam')}, val AUC={ck.get('val_auc')})")

# --- 2. A few MRNet validation exams ------------------------------------------
_, val_loader = build_dataloaders(
    task=bc.get("task", "acl"), plane=bc.get("plane", config.DEFAULT_PLANE),
    train_augment="none", batch_size=1, num_workers=0,
    output_size=bc.get("output_size", 256))
val_ds = val_loader.dataset

# --- 3. Grad-CAM++ on the most-attended slice of a few exams ------------------
# explain_exam() picks the slice from SliceAttentionPool's weights, then runs
# Grad-CAM++ on the backbone's last spatial map. Figures -> the interpretability
# stage folder (config.FIGURE_DIRS["interpretability"]).
target_layer = interp.get_target_layer(model, bc["backbone"])
fig_dir = config.FIGURE_DIRS["interpretability"]
fig_dir.mkdir(parents=True, exist_ok=True)

for idx in range(min(3, len(val_ds))):
    img, label = val_ds[idx]
    out = interp.explain_exam(model, img, target_layer, device=device, plusplus=True)
    sl = out["slice"]
    raw = img[sl, 0].numpy()                                  # one (H, W) slice
    overlay = interp.overlay_heatmap(raw, out["cam"][sl].numpy(), alpha=0.5)

    fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
    ax[0].imshow(raw, cmap="gray"); ax[0].set_title("raw MRI"); ax[0].axis("off")
    ax[1].imshow(overlay); ax[1].set_title("Grad-CAM++"); ax[1].axis("off")
    tear = "ACL tear" if int(label.view(-1).item()) == 1 else "intact"
    fig.suptitle(f"exam {idx} · true={tear} · p(tear)={out['prob']:.2f} · slice {sl}")
    fig.tight_layout()
    fig.savefig(fig_dir / f"example_{bc['backbone']}_exam{idx}.png", bbox_inches="tight")
    plt.show()

print("saved interpretability examples ->", fig_dir)
